# TFG lechugas · entrenamiento de la etapa 1 en Colab (GPU)

Este notebook **solo orquesta**: comprueba la GPU, monta Drive, deja el repositorio en un commit concreto, instala `requirements-train.txt` **sin tocar torch**, genera el dataset `etapa1_v1` en `/content` con el importador y entrena RF-DETR Nano (`python -m src.train.entrenar`). Toda la lógica está en `src/`; los parámetros, en `configs/train/etapa1_rfdetr_n.yaml`.

**Antes de empezar**
1. Entorno de ejecución → Cambiar tipo de entorno → GPU T4.
2. Sube los ZIP de Roboflow a `MyDrive/tfg-lechugas/data/raw` (los que usa `configs/datasets/etapa1_v1.yaml`). Los datos no van al repositorio.
3. En la celda de parámetros pon en `COMMIT` el commit exacto y en `EXP_ID` un nombre nuevo para la ejecución (no se pisa una ejecución anterior).
4. Antes del primer entrenamiento completo, ejecuta el modo `smoke` (celda g).

**Qué produce**
- `/content/TFG-VisionArtificial/data/processed/etapa1_v1`: el dataset (se genera aquí, no se copia a Drive).
- `/content/trabajo/<EXP_ID>`: la copia de trabajo del dataset (con las repeticiones de svodk). **No se copia a Drive.**
- `/content/TFG-VisionArtificial/runs/<EXP_ID>`: la ejecución (configuración, entorno, métricas de valid, mejor punto, tiempo y pesos). Se copia a `MyDrive/tfg-lechugas/runs/<EXP_ID>`.

In [ ]:
# a) Parámetros. Es lo único que se edita antes de ejecutar (y MODO, en la celda g).
import os
import shlex
import subprocess
import sys

REPO_URL = "https://github.com/diegocalvofm-ux/TFG-VisionArtificial.git"
COMMIT = "main"  # rama, etiqueta o hash de commit; para reproducir un entrenamiento, un hash
EXP_ID = "etapa1_rfdetr_n_001"  # carpeta runs/<EXP_ID>; cámbialo en cada ejecución
CARPETA_DRIVE = "/content/drive/MyDrive/tfg-lechugas"
ZIPS_DRIVE = f"{CARPETA_DRIVE}/data/raw"  # ZIP de Roboflow, fuera del repositorio
CONFIG = "configs/train/etapa1_rfdetr_n.yaml"

# Derivados: no hace falta tocarlos. Las rutas no deben llevar espacios.
REPO = "/content/TFG-VisionArtificial"
RUNS = f"{REPO}/runs"
TRABAJO = "/content/trabajo"  # copias de trabajo del dataset: en /content, nunca en Drive
PYTHON = sys.executable


def ejecutar(comando, cwd=None, comprobar=True):
    """Lanza un comando, enseña su salida según va saliendo y devuelve su código de salida.

    En lenguaje llano: es como escribir el comando en una terminal. Si el comando falla
    (código distinto de 0) lanza un error y el notebook se para en esa celda, en vez de seguir
    con un entorno a medias. Con comprobar=False solo devuelve el código.
    """
    print(f"$ {comando}", flush=True)
    entorno = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8"}
    proceso = subprocess.Popen(
        shlex.split(comando),
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        env=entorno,
    )
    for linea in proceso.stdout:
        print(linea, end="", flush=True)
    codigo = proceso.wait()
    if comprobar and codigo != 0:
        raise RuntimeError(f"El comando terminó con código {codigo}: {comando}")
    return codigo

In [ ]:
# b) GPU. Para aquí si el entorno no tiene GPU (sería un entrenamiento de horas en CPU).
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No hay GPU: Entorno de ejecución -> Cambiar tipo de entorno -> GPU T4")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# c) Drive. Monta tu Drive y comprueba que los ZIP de Roboflow están donde se espera.
import os

from google.colab import drive

drive.mount("/content/drive")

if not os.path.isdir(ZIPS_DRIVE):
    raise FileNotFoundError(
        f"No existe la carpeta de datos: {ZIPS_DRIVE}. Sube los ZIP de Roboflow a esa carpeta "
        "de Drive (o corrige ZIPS_DRIVE en la celda de parámetros) y vuelve a ejecutar."
    )
zips = sorted(n for n in os.listdir(ZIPS_DRIVE) if n.lower().endswith(".zip"))
if not zips:
    raise FileNotFoundError(f"No hay ZIP en {ZIPS_DRIVE}. Sube los ZIP de Roboflow a esa carpeta.")
print(f"{len(zips)} ZIP en {ZIPS_DRIVE}: " + ", ".join(zips))

In [ ]:
# d) Repositorio. Lo clona (o lo actualiza) y lo deja en el commit pedido.
import os

if os.path.isdir(os.path.join(REPO, ".git")):
    ejecutar("git fetch --all --tags", cwd=REPO)
else:
    ejecutar(f"git clone {REPO_URL} {REPO}")
ejecutar(f"git checkout {COMMIT}", cwd=REPO)
# Si COMMIT es una rama se pone al día; con un hash o una etiqueta no hay nada que actualizar.
if ejecutar("git symbolic-ref -q HEAD", cwd=REPO, comprobar=False) == 0:
    ejecutar("git pull --ff-only", cwd=REPO)
os.chdir(REPO)
ejecutar("git rev-parse --short HEAD")  # el commit exacto que se va a entrenar: apúntalo

In [ ]:
# e) Dependencias SIN tocar torch. Colab ya trae torch y torchvision: se guardan sus versiones, se
# instala con un constraints que las fija y se comprueba después que no cambiaron (si cambian, el
# notebook se para aquí).
ANTES = "/content/torch_antes.json"
DESPUES = "/content/torch_despues.json"
CONSTRAINTS = "/content/constraints.txt"

ejecutar(f"{PYTHON} -m src.train.entorno versiones --out {ANTES}", cwd=REPO)
ejecutar(f"{PYTHON} -m src.train.entorno constraints --out {CONSTRAINTS}", cwd=REPO)
ejecutar(f"{PYTHON} -m pip install -q -c {CONSTRAINTS} -r requirements.txt -r requirements-train.txt", cwd=REPO)
ejecutar(f"{PYTHON} -m src.train.entorno versiones --out {DESPUES}", cwd=REPO)
ejecutar(f"{PYTHON} -m src.train.entorno comprobar --antes {ANTES} --despues {DESPUES}", cwd=REPO)

In [ ]:
# f) Datos. Copia los ZIP de Drive a data/raw del repositorio clonado (en /content, donde los espera
# configs/datasets/*.yaml) y genera el dataset etapa1_v1 con el importador. Nada de esto va a Drive.
import os
import shutil

os.makedirs(f"{REPO}/data/raw", exist_ok=True)
for nombre in sorted(n for n in os.listdir(ZIPS_DRIVE) if n.lower().endswith(".zip")):
    shutil.copy2(f"{ZIPS_DRIVE}/{nombre}", f"{REPO}/data/raw/{nombre}")
    print(f"Copiado {nombre}")
ejecutar(
    f"{PYTHON} -m src.data.importar_publicos importar --etapa configs/datasets/etapa1_v1.yaml",
    cwd=REPO,
)

In [ ]:
# g) Modo. "smoke" = 1 época con unas 20 imágenes de train y todo valid (prueba rápida: hazla antes
# del primer entrenamiento completo). "completo" = el entrenamiento entero del YAML.
MODO = "smoke"  # "smoke" | "completo"

In [ ]:
# h) Entrenamiento. Escribe runs/<EXP_ID>/ y, al final, imprime la línea REGISTRO: y el comando
# para anotar el experimento. La copia de trabajo del dataset se crea en TRABAJO (no en runs/).
if MODO not in ("smoke", "completo"):
    raise ValueError(f'MODO debe ser "smoke" o "completo" (es {MODO!r})')
opcion_smoke = " --smoke" if MODO == "smoke" else ""
ejecutar(
    f"{PYTHON} -m src.train.entrenar --config {CONFIG} --exp-id {EXP_ID} --trabajo {TRABAJO}{opcion_smoke}",
    cwd=REPO,
)

In [ ]:
# i) Copia la ejecución a Drive (para no perderla al cerrarse la sesión de Colab). Solo va
# runs/<EXP_ID>: la copia de trabajo del dataset se queda en /content.
import os
import shutil

ORIGEN = f"{RUNS}/{EXP_ID}"
DESTINO = f"{CARPETA_DRIVE}/runs/{EXP_ID}"
shutil.copytree(ORIGEN, DESTINO, dirs_exist_ok=True)  # repetirla sobrescribe, no falla
for carpeta, _, archivos in sorted(os.walk(DESTINO)):
    for nombre in sorted(archivos):
        ruta = os.path.join(carpeta, nombre)
        kilobytes = os.path.getsize(ruta) / 1024
        print(f"{os.path.relpath(ruta, DESTINO)}  ({kilobytes:.1f} KB)")
print(f"Copiado en {DESTINO}")

## Después

1. Descarga la carpeta `MyDrive/tfg-lechugas/runs/<EXP_ID>` de Drive a `runs/<EXP_ID>` del repositorio local (la carpeta `runs/` no se sube a git).
2. Mira `runs/<EXP_ID>/resumen.json`: mejor punto (AP50 y AP50:95 de valid), tiempo de entrenamiento, memoria de la GPU y pesos en `pesos/`.
3. Si no era un `smoke`, anota el experimento con Claude Code. La salida de la celda h trae la línea que empieza por `REGISTRO:` (commit, GPU y versiones) y el comando `python -m src.experimentos.registro campos --exp expNNN --set ...` con los campos de la cabecera ya rellenos; pega la línea `REGISTRO:` en las observaciones y usa `/registrar-experimento`. `resumen.json` guarda los mismos campos. No se inventa ninguna métrica: si el entrenamiento no dejó métricas, el valor queda vacío.

El dataset de trabajo (`/content/trabajo/<EXP_ID>`, con las repeticiones de svodk) no se copia a Drive: se vuelve a crear en cada ejecución a partir de los ZIP.